# Project: Real-World Data Wrangling Pipeline

Build an end-to-end pipeline that cleans, transforms, and merges multiple messy datasets.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import pandas as pd
import numpy as np

print('Pandas version:', pd.__version__)

## Create Messy Datasets

In [ ]:
np.random.seed(42)
n = 500
orders = pd.DataFrame({
    'order_id': range(1001, 1001+n),
    'customer_id': np.random.choice(['CUST_'+str(i) for i in range(100, 200)], n),
    'order_date': pd.date_range('2023-01-01', periods=n, freq='D'),
    'amount': np.round(np.random.lognormal(4, 1, n), 2),
    'status': np.random.choice(['delivered','pending','cancelled','shipped','RETURNED','Delivered'], n),
    'product_code': np.random.choice(['PROD-A','PROD-B','PROD-c','prod-d',' PROD-E'], n),
})
orders.loc[np.random.choice(n, 20), 'customer_id'] = None
print('Orders:')
print(orders.head(), f'\nShape: {orders.shape}')

In [ ]:
customers = pd.DataFrame({
    'customer_id': ['CUST_'+str(i) for i in range(100, 200)],
    'name': ['Customer '+str(i) for i in range(100, 200)],
    'tier': np.random.choice(['Gold','Silver','Bronze','Platinum','gold','SILVER'], 100),
})
customers = pd.concat([customers, customers.iloc[[0,5,10]]]).reset_index(drop=True)
print('Customers:')
print(customers.head(), f'\nShape: {customers.shape}')

## Step 1-2: Clean Status and Product Codes

In [ ]:
orders['status_clean'] = orders['status'].str.strip().str.lower()\
    .map({'delivered':'Delivered','shipped':'Shipped','pending':'Pending',
          'cancelled':'Cancelled','returned':'Returned'})
orders['product_clean'] = orders['product_code'].str.strip().str.upper().str.replace(' ', '_')
print('Status values:', orders['status_clean'].unique())
print('Product values:', orders['product_clean'].unique())

## Step 3-4: Fix Amounts and Missing IDs

In [ ]:
orders['amount_clean'] = orders['amount'].abs()
neg_count = (orders['amount'] < 0).sum()
orders_clean = orders.dropna(subset=['customer_id']).copy()
print(f'Fixed {neg_count} negative amounts')
print(f'Dropped {len(orders)-len(orders_clean)} rows with missing customer_id')

## Step 5-6: Clean Customer Tiers and Deduplicate

In [ ]:
customers['tier_clean'] = customers['tier'].str.strip().str.capitalize()
tier_map = {'Platinum':'Platinum','Gold':'Gold','Silver':'Silver','Bronze':'Bronze'}
customers['tier_clean'] = customers['tier_clean'].map(tier_map)
print(f'Before dedup: {len(customers)}')
customers_deduped = customers.drop_duplicates(subset=['customer_id']).reset_index(drop=True)
print(f'After dedup: {len(customers_deduped)}')
print(f'Tier distribution:\n{customers_deduped["tier_clean"].value_counts()}')

## Step 7-8: Merge and Feature Engineering

In [ ]:
merged = orders_clean.merge(
    customers_deduped[['customer_id','name','tier_clean']],
    on='customer_id', how='left')
merged['order_year'] = pd.DatetimeIndex(merged['order_date']).year
merged['order_month'] = pd.DatetimeIndex(merged['order_date']).month
merged['order_dow'] = pd.DatetimeIndex(merged['order_date']).dayofweek
merged['is_weekend'] = merged['order_dow'].isin([5,6]).astype(int)
print(f'Merged: {merged.shape[0]} rows x {merged.shape[1]} cols')
print(merged[['order_id','order_year','order_month','tier_clean']].head())

## Step 9: Wrangling Summary

In [ ]:
print('='*60)
print('DATA WRANGLING SUMMARY'.center(60))
print('='*60)
print(f'\nInput orders: {n}')
print(f'Final clean rows: {len(merged)}')
print(f'Issues fixed: {neg_count} negative amounts, '
      f'{len(orders)-len(orders_clean)} missing IDs, '
      f'{len(customers)-len(customers_deduped)} duplicates')
print(f'Revenue: ${merged["amount_clean"].sum():,.2f}')
print(f'Avg order: ${merged["amount_clean"].mean():.2f}')

## Summary

- String cleaning and standardization
- Handling missing values
- Deduplication
- Table merging
- Feature engineering